# Heritage Short: Wan 2.2 TI2V-5B image-to-video (T4)
Pushed by `pipeline/shots.py` via `pipeline/kaggle_job.py`. The JOBS cell is filled in at push time
(clips to render, keyframe sha256, prompts, seeds). Keyframes arrive as the private dataset
`<user>/heritage-short-wan-in`, named by sha256 so a stale dataset version is detected.
The T4 has no bfloat16, so the transformer runs in float16 with model CPU offload; the VAE stays float32 (tiled).
Writes `out/<shot>.mp4` and `timing.jsonl` (GPU seconds per attempt) to /kaggle/working.

In [ ]:
# @JOBS (replaced at push time)
JOBS = {"job_id": "local-test", "config": {}, "jobs": []}

In [ ]:
import json, os, sys, time, glob, hashlib, subprocess, traceback
T0 = time.time()
OUT = "/kaggle/working"
os.makedirs(f"{OUT}/out", exist_ok=True)
_log = open(f"{OUT}/timing.jsonl", "a")
def log(**kw):
    """One JSON line per attempt; pipeline/kaggle_job.py copies these into runs.jsonl."""
    _log.write(json.dumps({"job_id": JOBS["job_id"], **kw}) + "\n"); _log.flush(); print(kw)

subprocess.run([sys.executable, "-m", "pip", "install", "-q", "diffusers==0.39.0", "ftfy", "imageio-ffmpeg"], check=True)

In [ ]:
import numpy as np, torch
from PIL import Image
from diffusers import AutoencoderKLWan, WanImageToVideoPipeline
from diffusers.utils import export_to_video

cfg = JOBS["config"]
vae = AutoencoderKLWan.from_pretrained(cfg["model"], subfolder="vae", torch_dtype=torch.float32)
pipe = WanImageToVideoPipeline.from_pretrained(cfg["model"], vae=vae, torch_dtype=torch.float16)
pipe.enable_model_cpu_offload()
if hasattr(pipe.vae, "enable_tiling"):
    pipe.vae.enable_tiling()
log(id=None, event="setup", seconds=round(time.time() - T0, 1), gpu=torch.cuda.get_device_name(0),
    torch=torch.__version__)

In [ ]:
def find_image(name, sha):
    for p in glob.glob(f"/kaggle/input/**/{name}", recursive=True):
        if hashlib.sha256(open(p, "rb").read()).hexdigest() == sha:
            return p
    return None

def fit(img, w, h):
    """Centre-crop to the target aspect, then resize (keyframes are 16:9, Wan wants w x h)."""
    a = w / h
    W, H = img.size
    if W / H > a:
        nw = round(H * a); img = img.crop(((W - nw) // 2, 0, (W - nw) // 2 + nw, H))
    else:
        nh = round(W / a); img = img.crop((0, (H - nh) // 2, W, (H - nh) // 2 + nh))
    return img.resize((w, h), Image.LANCZOS)

for job in JOBS["jobs"]:
    path = find_image(job["image"], job["image_sha256"])
    if path is None:
        log(id=job["id"], status="error", attempt=0, gpu_seconds=0, error="keyframe not found (stale dataset version?)")
        continue
    image = fit(Image.open(path).convert("RGB"), cfg["width"], cfg["height"])
    for attempt in range(1, job["max_attempts"] + 1):
        seed = job["seed"] + attempt - 1          # a retry uses the next seed
        torch.cuda.reset_peak_memory_stats()
        t = time.time()
        try:
            frames = pipe(image=image, prompt=job["prompt"], negative_prompt=cfg["negative_prompt"],
                          height=cfg["height"], width=cfg["width"], num_frames=job["frames"],
                          num_inference_steps=cfg["steps"], guidance_scale=cfg["guidance"],
                          generator=torch.Generator("cpu").manual_seed(seed), output_type="np").frames[0]
            if not np.isfinite(frames).all() or float(frames.mean()) < 0.02:
                raise RuntimeError("NaN or black frames (float16 overflow?)")
            export_to_video(list(frames), f"{OUT}/out/{job['id']}.mp4", fps=cfg["fps"])
            log(id=job["id"], status="ok", attempt=attempt, seed=seed, gpu_seconds=round(time.time() - t, 1),
                frames=len(frames), file=f"out/{job['id']}.mp4",
                peak_vram_gb=round(torch.cuda.max_memory_allocated() / 1e9, 2))
            break
        except Exception as e:
            log(id=job["id"], status="error", attempt=attempt, seed=seed, gpu_seconds=round(time.time() - t, 1),
                error=repr(e)[:400])
            traceback.print_exc()
            torch.cuda.empty_cache()
log(id=None, event="done", seconds=round(time.time() - T0, 1))